# 02 — Central K-Means Evaluation

Run centralized k-means clustering on before and after correction matrices.

**What this notebook does:**
1. Load prepared data from notebook 01.
2. Scale features (center + variance, matching the federated app).
3. Run k-means for each k value on both before and corrected data.
4. Align predicted clusters to true labels (condition, batch).
5. Compute metrics (ARI, MCC, accuracy, precision, recall, F1).
6. Save central k-means results.

## Imports

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evaluation_utils.real_datasets_utils import (
    dataset_configs,
    discover_clients,
    load_feature_matrix,
    load_metadata,
    run_central_kmeans,
    merge_central_clusters,
    evaluate_metrics,
    save_metrics_tables,
)

## Configuration

In [2]:
DATASETS = [
    "ecoli", 
    "ovarian_cancer", 
    "quartet", "ccRCC_proteomics", "multiomics"]
SEED = 11  # Random seed for reproducibility

# Where prepared data was saved by notebook 01.
OUTPUT_ROOT = NOTEBOOK_DIR

## Run Central K-Means

For each dataset, load the prepared before/corrected matrices,
run k-means, and compute metrics against true condition and batch labels.

In [3]:
all_metrics = []
configs = dataset_configs(REPO_ROOT)

for ds_name in DATASETS:
    cfg = configs[ds_name]
    ds_dir = OUTPUT_ROOT / ds_name / "prepared"

    # Load prepared matrices from notebook 01
    before = load_feature_matrix(ds_dir / "before_matrix.tsv")
    corrected = load_feature_matrix(ds_dir / "corrected_matrix.tsv")
    metadata = pd.read_csv(ds_dir / "metadata.tsv", sep="\t")

    k_condition = int(metadata['condition'].nunique())
    k_batch = int(metadata['lab'].nunique())
    k_values = sorted(set([k_condition, k_batch]))

    print(f"\n{'='*60}")
    print(f"{ds_name}: running central k-means with k={k_values}, seed={SEED}")
    print(f"{'='*60}")

    # Run central k-means on both matrices
    before_clusters = run_central_kmeans(before, k_values, SEED, n_init=cfg.n_init)
    corrected_clusters = run_central_kmeans(corrected, k_values, SEED, n_init=cfg.n_init)

    # Merge cluster assignments with metadata
    central_meta = merge_central_clusters(metadata, before_clusters, corrected_clusters)

    # Save central k-means results
    run_dir = OUTPUT_ROOT / ds_name / "kmeans_res" / "runs"
    run_dir.mkdir(parents=True, exist_ok=True)
    central_path = run_dir / "1_metadata_cntrl_kmeans_res.tsv"
    central_meta.to_csv(central_path, sep="\t", index=False)
    print(f"Saved: {central_path}")

    # Compute metrics (central only — no federated results yet)
    metrics = evaluate_metrics(
        dataset_name=ds_name,
        central_res=central_meta,
        before_fed_res=None,
        after_fed_res=None,
        k_condition=k_condition,
        k_batch=k_batch,
    )
    save_metrics_tables(metrics, OUTPUT_ROOT / ds_name / "metrics")
    all_metrics.append(metrics)
    print(f"\nMetrics for {ds_name}:")
    display(metrics[["Target", "Method", "ARI"]])


ecoli: running central k-means with k=[2, 5], seed=11
Saved: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/ecoli/kmeans_res/runs/1_metadata_cntrl_kmeans_res.tsv

Metrics for ecoli:


,Target,Method,ARI
0,condition,ecoli_condition_BC_Cntrl_2cls,-0.005105
1,condition,ecoli_condition_AC_Cntrl_2cls,1.000000
2,batch,ecoli_batch_BC_Cntrl_5cls,1.000000
3,batch,ecoli_batch_AC_Cntrl_5cls,0.000512



ovarian_cancer: running central k-means with k=[2, 6], seed=11
Saved: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/ovarian_cancer/kmeans_res/runs/1_metadata_cntrl_kmeans_res.tsv

Metrics for ovarian_cancer:


,Target,Method,ARI
0,condition,ovarian_cancer_condition_BC_Cntrl_2cls,0.108155
1,condition,ovarian_cancer_condition_AC_Cntrl_2cls,0.893182
2,batch,ovarian_cancer_batch_BC_Cntrl_6cls,0.974282
3,batch,ovarian_cancer_batch_AC_Cntrl_6cls,0.103274



quartet: running central k-means with k=[4], seed=11
Saved: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/quartet/kmeans_res/runs/1_metadata_cntrl_kmeans_res.tsv

Metrics for quartet:


,Target,Method,ARI
0,condition,quartet_condition_BC_Cntrl_4cls,-0.040838
1,condition,quartet_condition_AC_Cntrl_4cls,0.618178
2,batch,quartet_batch_BC_Cntrl_4cls,0.425101
3,batch,quartet_batch_AC_Cntrl_4cls,-0.027860



ccRCC_proteomics: running central k-means with k=[2, 3], seed=11
Saved: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/ccRCC_proteomics/kmeans_res/runs/1_metadata_cntrl_kmeans_res.tsv

Metrics for ccRCC_proteomics:


,Target,Method,ARI
0,condition,ccRCC_proteomics_condition_BC_Cntrl_2cls,-0.000270
1,condition,ccRCC_proteomics_condition_AC_Cntrl_2cls,0.864936
2,batch,ccRCC_proteomics_batch_BC_Cntrl_3cls,0.345435
3,batch,ccRCC_proteomics_batch_AC_Cntrl_3cls,0.006300


## Combined Central Metrics

In [4]:
# Merge metrics across all datasets and save combined tables.
combined = pd.concat(all_metrics, ignore_index=True)
save_metrics_tables(combined, OUTPUT_ROOT)
print(f"Combined metrics saved to {OUTPUT_ROOT / 'metrics_ari.tsv'}")
display(combined[["Dataset", "Target", "Method", "ARI"]])

Combined metrics saved to /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/metrics_ari.tsv


,Dataset,Target,Method,ARI
0,ecoli,condition,ecoli_condition_BC_Cntrl_2cls,-0.005105
1,ecoli,condition,ecoli_condition_AC_Cntrl_2cls,1.000000
2,ecoli,batch,ecoli_batch_BC_Cntrl_5cls,1.000000
3,ecoli,batch,ecoli_batch_AC_Cntrl_5cls,0.000512
4,ovarian_cancer,condition,ovarian_cancer_condition_BC_Cntrl_2cls,0.108155
5,ovarian_cancer,condition,ovarian_cancer_condition_AC_Cntrl_2cls,0.893182
6,ovarian_cancer,batch,ovarian_cancer_batch_BC_Cntrl_6cls,0.974282
7,ovarian_cancer,batch,ovarian_cancer_batch_AC_Cntrl_6cls,0.103274
8,quartet,condition,quartet_condition_BC_Cntrl_4cls,-0.040838
9,quartet,condition,quartet_condition_AC_Cntrl_4cls,0.618178
